# 🚗 DRIVER DROWSINESS DETECTION AND MONITORING SYSTEM
### B.Tech Artificial Intelligence and Machine Learning | Mini Project
**Technologies Used:** Python 3.11, OpenCV, MediaPipe Face Mesh, Scikit-Learn (Random Forest), Pygame Mixer, NumPy, Pandas

---
### 📋 Notebook Navigation & Features:
1. **Dependency Installation & Setup**
2. **MediaPipe Landmark Indices & Aspect Ratio Functions (EAR & MAR)**
3. **Audio Alert Setup (`funny_munni.mp3` via Pygame)**
4. **Webcam Testing**
5. **Dataset Inspection & Random Forest Model Training**
6. **Advanced Feature 1: Personalized Eye Calibration Protocol** 🎯
7. **Advanced Feature 2: Three-Level Risk Classification Engine** 🚨
8. **Real-Time Live Video Detection with Song Alerts** 🎵
9. **Incident History & Session Logging** 📊

## 📦 1. Install & Import Libraries
> **Step Explanation (Hinglish):** Yahan sabhi zaroori packages verify aur import kiye jaate hain (OpenCV, MediaPipe, Scikit-learn, Pygame).

In [ ]:
import sys, os, time, csv
from pathlib import Path
from datetime import datetime

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import pygame

print("Python Version:", sys.version)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("\u2705 Sabhi libraries successfully import ho gayi hain!")

## 📁 2. Project Paths & Folder Setup
> **Step Explanation (Hinglish):** Models, Data, aur Logs ke folders setup karte hain taaki joblib model aur events CSV organize rahein.

In [ ]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
LOG_DIR = BASE_DIR / "logs"

for d in [DATA_DIR, MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "features.csv"
MODEL_PATH = MODEL_DIR / "drowsiness_model.joblib"
LOG_PATH = LOG_DIR / "drowsiness_events.csv"
SONG_PATH = BASE_DIR / "funny_munni.mp3"

print(f"Trained Model Path: {MODEL_PATH} (Exists: {MODEL_PATH.exists()})")
print(f"Features Data Path: {DATA_PATH} (Exists: {DATA_PATH.exists()})")
print(f"Song File Path:     {SONG_PATH} (Exists: {SONG_PATH.exists()})")

## 🎵 3. Audio Alarm Engine (`funny_munni.mp3` via Pygame Mixer)
> **Step Explanation (Hinglish):** Pygame mixer initialize karke alert song load karte hain. Isse laptop speaker se loud clear audio bajta hai.

In [ ]:
if not pygame.mixer.get_init():
    pygame.mixer.init()

if SONG_PATH.exists():
    pygame.mixer.music.load(str(SONG_PATH))
    print(f"\u2705 Song loaded successfully: {SONG_PATH.name}")
    # Quick 1-second audio test
    print("Testing song playback for 1 second...")
    pygame.mixer.music.play(-1)
    time.sleep(1)
    pygame.mixer.music.stop()
    print("\u2705 Audio test successful! Song will loop during High Risk warnings.")
else:
    print(f"\u26a0\ufe0f Song file not found at {SONG_PATH}. Please place funny_munni.mp3 in the folder.")

## 👁️ 4. MediaPipe Landmark Indices & Biometric Formulas
> **Step Explanation (Hinglish):** MediaPipe 468 facial mesh se Left Eye, Right Eye aur Mouth ke key landmarks extract karke EAR aur MAR calculate karte hain.

In [ ]:
LEFT_EYE_INDICES = [362, 385, 387, 263, 373, 380]
RIGHT_EYE_INDICES = [33, 160, 158, 133, 153, 144]
MOUTH_INDICES = [61, 291, 39, 181, 0, 17, 269, 405]

def calculate_ear(landmarks, eye_indices, w, h):
    """Eye Aspect Ratio (EAR) = (vertical1 + vertical2) / (2 * horizontal)"""
    pts = np.array([[landmarks[i].x * w, landmarks[i].y * h] for i in eye_indices])
    A = np.linalg.norm(pts[1] - pts[5])
    B = np.linalg.norm(pts[2] - pts[4])
    C = np.linalg.norm(pts[0] - pts[3])
    return float((A + B) / (2.0 * C + 1e-6))

def calculate_mar(landmarks, mouth_indices, w, h):
    """Mouth Aspect Ratio (MAR) for detecting yawning."""
    pts = np.array([[landmarks[i].x * w, landmarks[i].y * h] for i in mouth_indices])
    A = np.linalg.norm(pts[2] - pts[6])
    B = np.linalg.norm(pts[3] - pts[5])
    C = np.linalg.norm(pts[0] - pts[1])
    return float((A + B) / (2.0 * C + 1e-6))

print("\u2705 EAR and MAR calculation functions defined.")

## 📊 5. Machine Learning Model Inspection & Feature Importance
> **Step Explanation (Hinglish):** Random Forest classifier (`drowsiness_model.joblib`) ko load karte hain aur uski feature importances check karte hain.

In [ ]:
model = None
feature_columns = ["left_ear", "right_ear", "avg_ear", "mar", "eye_closed_seconds"]

if MODEL_PATH.exists():
    bundle = joblib.load(MODEL_PATH)
    if isinstance(bundle, dict):
        model = bundle.get("model")
        feature_columns = bundle.get("feature_columns", feature_columns)
    else:
        model = bundle
    print(f"\u2705 Model Loaded: {type(model).__name__}")
    print(f"Target Classes:  {list(model.classes_)}")
    print(f"Features Used:   {feature_columns}")
    
    if hasattr(model, "feature_importances_"):
        fi = pd.DataFrame({"Feature": feature_columns, "Importance": model.feature_importances_})
        print("\n--- Feature Importances ---")
        print(fi.sort_values(by="Importance", ascending=False).to_string(index=False))
else:
    print("\u26a0\ufe0f Model file not found. Please train or place drowsiness_model.joblib in models/")

## 🎯 6. ADVANCED FEATURE 1: Personalized Eye Calibration Function
> **Step Explanation (Hinglish):** Har vyakti ke aakho ka shape alag hota hai. Yeh function 5 seconds tak normal open-eyes EAR collect karta hai aur formula use karke personal threshold set karta hai:
> `personal_threshold = baseline_ear * 0.70` (clipped between 0.18 and 0.30).

In [ ]:
def calibrate_driver_ear(cap, face_mesh, calibration_seconds=5.0):
    """
    Personalized Eye Calibration Protocol:
    Records natural open-eye EAR for 5 seconds and calculates a personalized threshold.
    """
    print("\n--- Starting Eye Calibration (Look naturally at camera) ---")
    start_time = time.time()
    ear_samples = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        now = time.time()
        elapsed = now - start_time
        progress = min(int((elapsed / calibration_seconds) * 100), 100)
        
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = face_mesh.process(rgb)
        
        if res.multi_face_landmarks:
            lm = res.multi_face_landmarks[0].landmark
            l_ear = calculate_ear(lm, LEFT_EYE_INDICES, w, h)
            r_ear = calculate_ear(lm, RIGHT_EYE_INDICES, w, h)
            ear_samples.append((l_ear + r_ear) / 2.0)
        
        # HUD Calibration bar
        cv2.rectangle(frame, (0, 0), (w, 50), (20, 25, 35), -1)
        cv2.putText(frame, f"CALIBRATING EYES: {progress}% (Keep looking naturally)", (15, 32),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 80), 2)
        cv2.imshow("Driver Calibration Studio", frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q') or elapsed >= calibration_seconds:
            break
            
    cv2.destroyWindow("Driver Calibration Studio")
    
    if len(ear_samples) > 10:
        baseline_ear = float(np.percentile(ear_samples, 80))
        personal_thresh = float(np.clip(baseline_ear * 0.70, 0.18, 0.30))
    else:
        baseline_ear = 0.35
        personal_thresh = 0.25
        
    print(f"\u2705 Calibration Completed!")
    print(f"   Baseline Open-Eye EAR: {baseline_ear:.3f}")
    print(f"   Calculated Personal Threshold: {personal_thresh:.3f} (Baseline * 0.70)")
    return personal_thresh, baseline_ear

print("\u2705 Calibration function ready.")

## 🚨 7. ADVANCED FEATURE 2: Three-Level Risk Classification Function
> **Step Explanation (Hinglish):** Binary classifier ke upar ek 3-Level safety decision layer banate hain:
> - **Level 1 (NORMAL):** Green status, attentive driver.
> - **Level 2 (CAUTION):** Yellow status, partial closure ya yawning (MAR >= 0.62).
> - **Level 3 (HIGH RISK):** Red status, continuous closure >= 2.0 seconds -> `funny_munni.mp3` baje ga!

In [ ]:
def classify_risk_level(avg_ear, mar, closed_seconds, ml_prediction, ml_confidence, threshold):
    """
    Three-Level Risk Decision Layer:
    1. HIGH_RISK: Sustained eye closure >= 0.5s (Aankh band hote hi funny_munni.mp3 song baje!)
    2. CAUTION: Yawning (MAR >= 0.62) ya partial eye closure (0.2s).
    3. NORMAL: Attentive driver with open eyes.
    """
    if closed_seconds >= 0.5 or (ml_prediction.lower() == "drowsy" and closed_seconds >= 0.3):
        return "HIGH_RISK"
    elif (closed_seconds >= 0.2) or (mar >= 0.62):
        return "CAUTION"
    else:
        return "NORMAL"


## 🎬 8. Real-Time Detection with Personalized Calibration & Song Alerts
> **Step Explanation (Hinglish):** Yeh main real-time loop hai. Pehle 5-second calibration hogi, phir live detection chalegi. Jab driver ki aakhein 2 seconds band hongi, `funny_munni.mp3` bajna shuru ho jayega aur aakhein kholne par ruk jayega. Press **Q** to exit.

In [ ]:
def run_full_drowsiness_system(camera_index=0):
    if model is None:
        print("\u26a0\ufe0f Model not loaded. Please train or load the model first.")
        return
        
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        cap = cv2.VideoCapture(camera_index, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print(f"\u26a0\ufe0f Cannot open camera Device {camera_index}.")
        return

    mp_mesh = mp.solutions.face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )

    # STEP 1: Run Personalized Eye Calibration
    personal_threshold, baseline_ear = calibrate_driver_ear(cap, mp_mesh, calibration_seconds=5.0)
    
    # STEP 2: Main Monitoring Loop
    print("\n--- Live Monitoring Active (Press 'Q' in the video window to stop) ---")
    
    eye_close_start = None
    last_drowsy_time = 0.0
    events_logged = []
    alert_debounce = False
    
    risk_colors = {
        "NORMAL":    (0, 200, 80),    # Green
        "CAUTION":   (0, 165, 255),   # Orange
        "HIGH_RISK": (0, 0, 240),     # Red
    }

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
                
            frame = cv2.flip(frame, 1)
            h, w = frame.shape[:2]
            now = time.time()
            
            # IMPORTANT FIX: Always initialize closed_seconds to avoid NameError!
            closed_seconds = 0.0
            left_ear = right_ear = avg_ear = mar = 0.0
            ml_pred = "Alert"
            ml_conf = 0.5
            risk = "NORMAL"
            status_text = "FACE SEARCHING"
            
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = mp_mesh.process(rgb)
            
            if res.multi_face_landmarks:
                lm = res.multi_face_landmarks[0].landmark
                left_ear  = calculate_ear(lm, LEFT_EYE_INDICES, w, h)
                right_ear = calculate_ear(lm, RIGHT_EYE_INDICES, w, h)
                avg_ear   = (left_ear + right_ear) / 2.0
                mar       = calculate_mar(lm, MOUTH_INDICES, w, h)
                
                # Eye Closure Timer
                if avg_ear < personal_threshold:
                    if eye_close_start is None:
                        eye_close_start = now
                    closed_seconds = now - eye_close_start
                else:
                    eye_close_start = None
                    closed_seconds = 0.0
                    alert_debounce = False
                
                # Random Forest Inference
                feats = np.array([[left_ear, right_ear, avg_ear, mar, closed_seconds]])
                raw_p = model.predict(feats)[0]
                proba = model.predict_proba(feats)[0]
                classes = list(model.classes_)
                ml_pred = str(raw_p)
                idx = classes.index(ml_pred) if ml_pred in classes else 0
                ml_conf = float(proba[idx])
                
                # 3-Level Risk Classification
                risk = classify_risk_level(avg_ear, mar, closed_seconds, ml_pred, ml_conf, personal_threshold)
                
                # Audio Control: Play funny_munni.mp3 on HIGH_RISK
                if risk == "HIGH_RISK":
                    last_drowsy_time = now
                    if not pygame.mixer.music.get_busy():
                        pygame.mixer.music.play(-1)
                        
                    if not alert_debounce:
                        events_logged.append({
                            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                            "event": "HIGH_RISK_DROWSY",
                            "ml_prediction": ml_pred,
                            "avg_ear": round(avg_ear, 3),
                            "mar": round(mar, 3),
                            "closed_sec": round(closed_seconds, 1)
                        })
                        alert_debounce = True
                        
                elif (now - last_drowsy_time) >= 0.4 and pygame.mixer.music.get_busy():
                    pygame.mixer.music.stop()
                
                # Draw Landmark circles
                for i in LEFT_EYE_INDICES + RIGHT_EYE_INDICES:
                    cv2.circle(frame, (int(lm[i].x * w), int(lm[i].y * h)), 2, (0, 255, 0), -1)
                for i in MOUTH_INDICES:
                    cv2.circle(frame, (int(lm[i].x * w), int(lm[i].y * h)), 2, (0, 165, 255), -1)
            else:
                eye_close_start = None
                if pygame.mixer.music.get_busy():
                    pygame.mixer.music.stop()
            
            # Visual HUD Rendering
            color = risk_colors.get(risk, (200, 200, 200))
            cv2.rectangle(frame, (0, 0), (w, 50), (15, 18, 25), -1)
            cv2.putText(frame, f"[{risk}] ML: {ml_pred} ({int(ml_conf*100)}%)", (15, 34),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2)
            
            hud_sub = f"EAR: {avg_ear:.3f} (Thresh: {personal_threshold:.3f}) | MAR: {mar:.3f} | Closed: {closed_seconds:.1f}s"
            cv2.rectangle(frame, (0, h - 35), (w, h), (15, 18, 25), -1)
            cv2.putText(frame, hud_sub, (12, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)
            
            if risk == "HIGH_RISK":
                cv2.putText(frame, "ALARM SONG ACTIVE! WAKE UP!", (w//2 - 160, h//2),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            
            cv2.imshow("Driver Drowsiness Detection System", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        pygame.mixer.music.stop()
        cap.release()
        cv2.destroyAllWindows()
        mp_mesh.close()
        
    if events_logged:
        ev_df = pd.DataFrame(events_logged)
        print(f"\n\u2705 Detection session finished. Logged {len(ev_df)} drowsiness event(s):")
        print(ev_df.to_string(index=False))
    else:
        print("\n\u2705 Detection session finished. No dangerous drowsiness events detected.")



### 🚀 Launch the Detector (Webcam Live)
> **Instructions:** Run the cell below. A camera window will appear. It will first calibrate for 5 seconds, then begin monitoring. Press **Q** in the window to stop.

In [ ]:
# To start live detection, uncomment and run:
# run_full_drowsiness_system(camera_index=0)